In [16]:
import sys
project_root = "/home/krosen5/Court Case Sequence Transformer Model/court_case_transformer_project"

if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [17]:
import pickle
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

In [18]:
BASE_CACHE = Path('/data/krosen5/cache/transformers/balanced_8')
MODELS_DIR = BASE_CACHE / 'saved_models'
OUTPUT_DIR = BASE_CACHE / 'error_analysis'
TARGET     = 'DOMESTIC_STAY_AWAY_ORDER_IMPOSED'

In [19]:
def load_pickle(path):
    with open(path, 'rb') as f:
        return pickle.load(f)
 
def get_text(proc):
    for k in ('text', 'proceeding_text', 'mag_text', 'cdc_text', 'notes'):
        v = proc.get(k, '')
        if v and str(v).strip():
            return str(v).strip()
    return '[no text]'
 
def get_ids(proc):
    return {
        'proceeding_id': proc.get('proceeding_id', ''),
        'case_id':       proc.get('case_id', '') or proc.get('docket_id', ''),
        'mag_num':       proc.get('mag_num', ''),
        'defendant_id':  proc.get('defendant_id', ''),
        'date':          str(proc.get('date', '')),
        'proc_type':     proc.get('type', ''),
    }

In [20]:
PALETTE = {
    'TP':          ('1F6B3A', 'E8F5E9'),
    'FP':          ('8B0000', 'FDECEA'),
    'FN':          ('7B4F00', 'FFF8E1'),
    'Qualitative': ('4A235A', 'F5EEF8'),
    'Summary':     ('1F4E79', 'DCE6F1'),
}
WIDE = {
    'proceeding_text': 80, 'all_proceedings_text': 90,
    'prior_proceeding_text': 70, 'true_labels': 50,
    'predicted_labels': 50, 'false_positives': 50,
    'false_negatives': 50, 'correct_labels': 50,
    'error_type': 55, 'wrong_label': 35,
    'true_labels_str': 50, 'predicted_labels_str': 50,
}
 
def write_wb(rows, path, key, title):
    if not rows:
        print(f"  ⚠️  {key}: no rows, skipping")
        return
    hdr, alt = PALETTE[key]
    wb = Workbook(); ws = wb.active; ws.title = key
    cols = list(rows[0].keys())
    # title banner
    ws.merge_cells(f'A1:{get_column_letter(len(cols))}1')
    ws['A1'] = title
    ws['A1'].font = Font(name='Arial', bold=True, size=13, color=hdr)
    ws['A1'].alignment = Alignment(horizontal='center')
    ws.row_dimensions[1].height = 24
    ws.append([])
    # header
    ws.append(cols)
    hr = ws.max_row
    for c, col in enumerate(cols, 1):
        cell = ws.cell(row=hr, column=c)
        cell.font = Font(name='Arial', bold=True, color='FFFFFF', size=10)
        cell.fill = PatternFill('solid', start_color=hdr)
        cell.alignment = Alignment(horizontal='center')
    ws.freeze_panes = ws.cell(row=hr+1, column=1)
    # data
    for i, row in enumerate(rows):
        ws.append([row.get(c, '') for c in cols])
        dr = ws.max_row
        af = PatternFill('solid', start_color=alt) if i % 2 == 1 else None
        for c, col in enumerate(cols, 1):
            cell = ws.cell(row=dr, column=c)
            cell.font = Font(name='Arial', size=9)
            if af: cell.fill = af
            if col in WIDE:
                cell.alignment = Alignment(wrap_text=True, vertical='top')
        ws.row_dimensions[dr].height = 80
    # widths
    for c, col in enumerate(cols, 1):
        ws.column_dimensions[get_column_letter(c)].width = WIDE.get(col, 16)
    wb.save(path)
    print(f"  ✅ {key:12s} {len(rows):4d} rows → {path.name}")

In [21]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('=' * 70)
print(f'  Error Analysis: {TARGET}')
print('=' * 70)

mr = load_pickle(MODELS_DIR / 'model_results.pkl')
fold_results  = mr['cv_training_results']['fold_results']   # list of 5 fold dicts
case_lvl_dict = mr['case_level_dict']                       # {0..4: [case dicts]}
misclass_dict = mr['confident_misclass_results_per_fold']   # {0..4: {...}}

# ── PART 1: case-level TP / FP / FN from case_level_dict ─────────────────
# Each fold's val_data is a list of (sequence, labels_raw) tuples.
# case_level_dict[fold_idx][i] corresponds to val_data[fold_idx][i].

tp_rows, fp_rows, fn_rows = [], [], []

for fold_idx, fold_data in enumerate(fold_results):
    val_data   = fold_data['val_data']          # list of (seq, labels_raw)
    case_dicts = case_lvl_dict[fold_idx]        # list of case result dicts

    print(f'\n── Fold {fold_idx+1}  ({len(val_data)} val cases) ──────────────')

    fold_tp = fold_fp = fold_fn = 0

    for case_dict in case_dicts:
        case_idx = case_dict['case_idx']

        in_tp = TARGET in case_dict.get('correct',          [])
        in_fp = TARGET in case_dict.get('false_positives',  [])
        in_fn = TARGET in case_dict.get('false_negatives',  [])

        if not (in_tp or in_fp or in_fn):
            continue

        # Pull sequence text from val_data
        if case_idx >= len(val_data):
            continue
        seq_item = val_data[case_idx]
        # val_data items are (sequence, labels_raw) tuples
        if isinstance(seq_item, (tuple, list)) and len(seq_item) == 2:
            sequence, labels_raw = seq_item
        else:
            sequence = seq_item
            labels_raw = []

        # Build one text block per proceeding in this case
        proc_texts = []
        for i, proc in enumerate(sequence):
            text = get_text(proc)
            ids  = get_ids(proc)
            proc_texts.append(
                f"[Step {i} | {ids['date']} | {ids['proc_type']}]\n{text}"
            )
        all_text = '\n\n---\n\n'.join(proc_texts)

        # Most recent proceeding (most likely where stay-away happened)
        last_proc = sequence[-1] if sequence else {}
        ids = get_ids(last_proc)
        prev_proc = sequence[-2] if len(sequence) >= 2 else {}

        base_row = {
            'fold':                  fold_idx + 1,
            'case_idx':              case_idx,
            'num_proceedings':       len(sequence),
            **ids,
            # Full text of every proceeding in this case (for reading)
            'all_proceedings_text':  all_text,
            # Most recent two proceedings (quickest to read)
            'last_proceeding_text':  get_text(last_proc),
            'prior_proceeding_text': get_text(prev_proc),
            # Label context
            'true_labels':           ' | '.join(case_dict.get('true_labels',      [])),
            'predicted_labels':      ' | '.join(case_dict.get('pred_labels',      [])),
            'correct_labels':        ' | '.join(case_dict.get('correct',          [])),
            'false_positives':       ' | '.join(case_dict.get('false_positives',  [])),
            'false_negatives':       ' | '.join(case_dict.get('false_negatives',  [])),
            'case_f1':               round(case_dict.get('f1', 0), 4),
            'case_precision':        round(case_dict.get('precision', 0), 4),
            'case_recall':           round(case_dict.get('recall', 0), 4),
        }

        if in_tp:
            tp_rows.append({**base_row, 'category': 'TP'})
            fold_tp += 1
        if in_fp:
            fp_rows.append({**base_row, 'category': 'FP'})
            fold_fp += 1
        if in_fn:
            fn_rows.append({**base_row, 'category': 'FN'})
            fold_fn += 1

    print(f'   Case-level TP={fold_tp}  FP={fold_fp}  FN={fold_fn}')

# ── PART 2: confident misclassification rows for stay-away ────────────────
# mistakes_report has one row per confident wrong prediction step.
# interpreted_mistakes may have richer text.
misclass_rows = []

for fold_idx, misclass_data in misclass_dict.items():
    for df_key in ('interpreted_mistakes', 'mistakes_report'):
        df = misclass_data.get(df_key)
        if df is None or not isinstance(df, pd.DataFrame) or df.empty:
            continue

        # Filter to stay-away related rows
        # The 'wrong_label' column is what the model predicted wrongly
        # Rows where wrong_label == TARGET → model falsely predicted stay-away (FP type)
        # We also want rows where TARGET appears in true_labels but wasn't predicted
        mask = pd.Series([False] * len(df))
        if 'wrong_label' in df.columns:
            mask |= df['wrong_label'].astype(str).str.contains(TARGET, na=False)
        if 'true_labels' in df.columns:
            mask |= df['true_labels'].astype(str).str.contains(TARGET, na=False)

        filtered = df[mask].copy()
        if filtered.empty:
            continue

        filtered.insert(0, 'fold', fold_idx + 1)
        filtered.insert(1, 'source', df_key)
        misclass_rows.append(filtered)
        print(f'   Fold {fold_idx+1} {df_key}: {len(filtered)} stay-away rows')
        break   # use the first non-empty df found

# ── PART 3: qualitative sheet (top examples for the paper) ────────────────
# Combine FP + FN case rows, sort by case_f1 ascending (worst cases first)
qual_rows = []
for r in fp_rows:
    r2 = dict(r)
    r2['error_type'] = 'FALSE POSITIVE — model predicted stay-away but it was not imposed'
    qual_rows.append(r2)
for r in fn_rows:
    r2 = dict(r)
    r2['error_type'] = 'FALSE NEGATIVE — stay-away was imposed but model missed it'
    qual_rows.append(r2)
qual_rows.sort(key=lambda r: r.get('case_f1', 1.0))  # worst F1 cases first

# Reorder columns: error type + text up front
if qual_rows:
    priority = ['error_type', 'category', 'fold', 'case_f1',
                'last_proceeding_text', 'prior_proceeding_text',
                'true_labels', 'predicted_labels',
                'false_positives', 'false_negatives', 'correct_labels',
                'all_proceedings_text',
                'num_proceedings', 'case_idx', 'date',
                'proceeding_id', 'case_id', 'mag_num', 'defendant_id',
                'proc_type', 'case_precision', 'case_recall']
    all_keys  = list(qual_rows[0].keys())
    remaining = [k for k in all_keys if k not in priority]
    ordered   = [k for k in priority if k in all_keys] + remaining
    qual_rows = [{k: r.get(k, '') for k in ordered} for r in qual_rows]

# ── WRITE FILES ───────────────────────────────────────────────────────────
print(f'\n── Writing Excel files {"─"*47}')

write_wb(tp_rows, OUTPUT_DIR / 'stay_away_TP.xlsx', 'TP',
            f'{TARGET} — True Positives (all proceedings in case shown)')
write_wb(fp_rows, OUTPUT_DIR / 'stay_away_FP.xlsx', 'FP',
            f'{TARGET} — False Positives (model predicted; label was NOT true)')
write_wb(fn_rows, OUTPUT_DIR / 'stay_away_FN.xlsx', 'FN',
            f'{TARGET} — False Negatives (label WAS true; model missed it)')
write_wb(qual_rows, OUTPUT_DIR / 'stay_away_qualitative.xlsx', 'Qualitative',
            f'Qualitative Error Examples — {TARGET} (worst F1 cases first)')

# Confident misclassification sheet
if misclass_rows:
    combined_mc = pd.concat(misclass_rows, ignore_index=True)
    mc_path = OUTPUT_DIR / 'stay_away_confident_misclass.xlsx'
    combined_mc.to_excel(mc_path, index=False)
    print(f"  ✅ Misclass      {len(combined_mc):4d} rows → {mc_path.name}")
else:
    print("  ⚠️  No confident misclassification rows found for stay-away")

print(f'\n{"="*70}')
print(f'  Saved to: {OUTPUT_DIR}')
print(f'  TP={len(tp_rows)}  FP={len(fp_rows)}  FN={len(fn_rows)}')
print(f'\n  READ FOR PAPER → stay_away_qualitative.xlsx')
print(f'  Sorted worst F1 first. Key columns:')
print(f'    last_proceeding_text   — the most recent hearing in the case')
print(f'    prior_proceeding_text  — the hearing before that')
print(f'    all_proceedings_text   — full case history (every step)')
print(f'    false_positives        — labels wrongly predicted in this case')
print(f'    false_negatives        — labels missed in this case')
print(f'    true_labels            — everything that was actually true')
print('=' * 70)

  Error Analysis: DOMESTIC_STAY_AWAY_ORDER_IMPOSED

── Fold 1  (507 val cases) ──────────────
   Case-level TP=62  FP=26  FN=42

── Fold 2  (507 val cases) ──────────────
   Case-level TP=75  FP=46  FN=29

── Fold 3  (507 val cases) ──────────────
   Case-level TP=82  FP=45  FN=32

── Fold 4  (506 val cases) ──────────────
   Case-level TP=79  FP=43  FN=30

── Fold 5  (506 val cases) ──────────────
   Case-level TP=71  FP=41  FN=33
   Fold 1 interpreted_mistakes: 38 stay-away rows
   Fold 2 interpreted_mistakes: 35 stay-away rows
   Fold 3 interpreted_mistakes: 40 stay-away rows
   Fold 4 interpreted_mistakes: 31 stay-away rows
   Fold 5 interpreted_mistakes: 29 stay-away rows

── Writing Excel files ───────────────────────────────────────────────
  ✅ TP            369 rows → stay_away_TP.xlsx
  ✅ FP            201 rows → stay_away_FP.xlsx
  ✅ FN            166 rows → stay_away_FN.xlsx
  ✅ Qualitative   367 rows → stay_away_qualitative.xlsx
  ✅ Misclass       173 rows → stay_away_confi